In [ ]:
import subprocess, sys, os

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Installing dependencies...")
_install("numpy<2.0")
_install("stable-baselines3[extra]>=2.3.0")
_install("gymnasium[atari,accept-rom-license]>=0.29.0")
_install("ale-py>=0.9.0")
_install("shimmy[atari]>=0.2.1")
_install("autorom[accept-rom-license]>=0.6.1")

print("Setting up AutoROM...")
subprocess.call(["AutoROM", "--accept-license", "-q"])

print("\nDependencies installed. Restarting runtime to apply numpy version...")
os.kill(os.getpid(), 9)

Installing dependencies...
Setting up AutoROM...


In [1]:
import os
import time
import warnings
import urllib.request
import numpy as np
import ale_py
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv, VecTransposeImage

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

GITHUB_REPO   = "Mugisha-isaac/Formative3-Group12-DQN"
GITHUB_BRANCH = "isaacm"
MODEL_NAME    = "dqn_model_exp10"
MODEL_ZIP     = f"{MODEL_NAME}.zip"
GITHUB_URL    = f"https://github.com/{GITHUB_REPO}/raw/{GITHUB_BRANCH}/models/{MODEL_ZIP}"

ENV_ID     = "ALE/Breakout-v5"
N_STACK    = 4
N_EPISODES = 5
STEP_DELAY = 0.02


def download_model():
    if os.path.exists(MODEL_ZIP):
        print(f"Model already exists: {MODEL_ZIP}")
        return
    print(f"Downloading {MODEL_ZIP} from GitHub ...")
    try:
        urllib.request.urlretrieve(GITHUB_URL, MODEL_ZIP)
        print(f"Downloaded -> {MODEL_ZIP}")
    except Exception as e:
        print(f"Download failed: {e}")
        print("Make sure the .zip file is uploaded to GitHub under models/ folder.")
        raise


def build_env():
    def _init():
        env = gym.make(ENV_ID, render_mode="human")
        env = AtariWrapper(env)
        return env
    env = DummyVecEnv([_init])
    env = VecFrameStack(env, n_stack=N_STACK)
    env = VecTransposeImage(env)
    return env


def play():
    download_model()

    print(f"\nLoading model: {MODEL_ZIP}")
    model = DQN.load(MODEL_ZIP)

    env = build_env()
    all_rewards = []
    all_lengths = []

    for ep in range(1, N_EPISODES + 1):
        obs       = env.reset()
        ep_reward = 0.0
        ep_length = 0
        done      = False

        print(f"\nEpisode {ep}/{N_EPISODES}")

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done_arr, _ = env.step(action)
            ep_reward += float(reward[0])
            ep_length += 1
            done = bool(done_arr[0])
            env.render()
            time.sleep(STEP_DELAY)

        all_rewards.append(ep_reward)
        all_lengths.append(ep_length)
        print(f"  Reward: {ep_reward:.1f}  |  Length: {ep_length} steps")

    env.close()

    print("\n" + "=" * 40)
    print(f"  Mean Reward : {np.mean(all_rewards):.2f}")
    print(f"  Best Episode: {np.max(all_rewards):.2f}")
    print(f"  Mean Length : {np.mean(all_lengths):.0f} steps")
    print("=" * 40)


if __name__ == "__main__":
    play()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Downloaded -> dqn_model_exp10.zip

Loading model: dqn_model_exp10.zip

Episode 1/5
  Reward: 4.0  |  Length: 41 steps

Episode 2/5
  Reward: 1.0  |  Length: 16 steps

Episode 3/5
  Reward: 2.0  |  Length: 28 steps

Episode 4/5
  Reward: 0.0  |  Length: 2 steps

Episode 5/5
  Reward: 0.0  |  Length: 2 steps

  Mean Reward : 1.40
  Best Episode: 4.00
  Mean Length : 18 steps
